# Difference-in-Differences: A Practitioner's Guide (Python replication)

This notebook replicates the empirical example in Baker, Callaway, Cunningham, Goodman-Bacon, and Sant'Anna (2026), "[Difference-in-Differences Designs: A Practitioner's Guide](https://psantanna.com/files/DiD_JEL.pdf)," *Journal of Economic Literature* 64(2): 498–557. The authors' R and Stata code is at [github.com/pedrohcgs/JEL-DiD](https://github.com/pedrohcgs/JEL-DiD).

Section numbers below follow the paper. The paper's organizing idea is that every DiD design, however complicated, is built from **2×2 building blocks**. For each one we are explicit about:

1. the **target parameter**: the causal quantity we want to learn;
2. the **identifying assumptions**: what must be true to recover it from data;
3. the **estimand**: the target parameter written in terms of observable population quantities;
4. the **estimator**: the sample analog we actually compute.

## Setup

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

import statsmodels.api as sm 
import statsmodels.formula.api as smf 

import pyfixest as pf

from tabulate import tabulate

## 2. Medicaid and Mortality: The Running Example

**Question:** How did the expansion of public health insurance (Medicaid) under the Affordable Care Act (ACA) affect adult mortality?

The ACA originally required every state to expand Medicaid eligibility in 2014, but a 2012 Supreme Court decision made expansion optional. States therefore expanded at different times, and some never did. The unit of observation is a county, observed each year from 2009 to 2019.

Notation from the paper:

| Symbol | Meaning | Column in `df` |
|---|---|---|
| $Y_{i,t}$ | crude mortality rate for adults ages 20–64, per 100,000, in county $i$ and year $t$ | `crude_rate` |
| $G_i$ | year county $i$'s state expanded Medicaid ($G_i = \infty$ if it never did) | `treat_year` (0 = not expanded by 2019) |
| $W_i$ | county's adult (20–64) population in 2013; used as the weights $\omega$ | `set_wt` |
| $X_{i,t}$ | covariates: % female, % White, % Hispanic, unemployment rate, poverty rate, median income (\$1,000s) | `perc_female`, `perc_white`, `perc_hispanic`, `unemp_rate_pc`, `poverty_rate`, `median_income_k` |

The cleaning below follows the authors' code. It drops DC and the states that expanded before 2014 (DE, MA, NY, VT), keeps counties with nonmissing covariates and a complete 2009–2019 panel, and recodes states that expanded after 2019 as untreated.

In [ ]:
# read in data
df = pd.read_csv('https://raw.githubusercontent.com/pedrohcgs/JEL-DiD/main/data/county_mortality_data.csv')

# drop 
drop_bool = df['state'].isin(["District of Columbia", "Delaware", "Massachusetts", "New York", "Vermont"])
# get index of indices to drop
drop_idx = df.loc[drop_bool].index
df.drop(index=drop_idx, inplace=True)

df['treat_year'] = df['yaca']
df['treat_year'] = df['treat_year'].fillna(0).astype(int)

# Destring covariates
cols = ['deaths', 'population_20_64', 'crude_rate_20_64', 'population_total',
        'population_20_64_hispanic', 'population_20_64_female', 'population_20_64_white',
        'unemployed', 'labor_force', 'unemp_rate', 'poverty_rate', 'median_income']
df[cols] = df[cols].apply(pd.to_numeric, errors='coerce')
df = df.rename(columns={'population_total': 'total_population'})

# Build covariates and drop missing
df['perc_white']      = df['population_20_64_white']    / df['population_20_64'] * 100
df['perc_hispanic']   = df['population_20_64_hispanic'] / df['population_20_64'] * 100
df['perc_female']     = df['population_20_64_female']   / df['population_20_64'] * 100
df['unemp_rate_pc']   = df['unemp_rate'] * 100
df['median_income_k'] = df['median_income'] / 1000
df = df.dropna(subset=['crude_rate_20_64', 'population_20_64', 'perc_white', 'perc_hispanic',
                       'perc_female', 'unemp_rate_pc', 'poverty_rate', 'median_income_k'])

# Keep counties with full 11-year panel
df = df[df.groupby('county_code')['year'].transform('size') == 11]
df.loc[df['treat_year'] > 2019, 'treat_year'] = 0

# Population weight (2013 population, per county)
df['set_wt'] = (df['population_20_64'].where(df['year'] == 2013)
                .groupby(df['county_code']).transform('max'))

df.rename(columns={'crude_rate_20_64': 'crude_rate'}, inplace=True)

## 3. 2×2 DiD Designs

We use two periods, 2013 (pre) and 2014 (post), and two groups:

- **treated:** counties in states that expanded in 2014 ($G_i = 2014$, so $D_i = 1$);
- **comparison:** counties in states that had not expanded by 2019 ($G_i > 2019$, so $D_i = 0$).

The treatment status dummy is $D_{i,t} = D_i \times \mathbb{1}\{t \geq 2014\}$.

The paper writes weighted expectations as $\mathbb{E}_\omega[A \mid C] = \mathbb{E}[\omega A \mid C] \,/\, \mathbb{E}[\omega \mid C]$. When $\omega = 1$ this is the ordinary conditional expectation. **Sections 3 and 4 of this notebook reproduce the paper's unweighted results ($\omega = 1$).**

### 3.1 Target parameter: the average treatment effect on the treated, $ATT(t)$

Let $Y_{i,t}(1)$ be county $i$'s mortality rate in year $t$ if its state expanded Medicaid in 2014, and $Y_{i,t}(0)$ its mortality rate if the state did not. We only ever observe one of them:

$$Y_{i,t} = (1 - D_i)\,Y_{i,t}(0) + D_i\,Y_{i,t}(1). \tag{1}$$

The target parameter is the **average treatment effect on the treated** in period $t$:

$$ATT(t) = \mathbb{E}_\omega[Y_{i,t}(1) - Y_{i,t}(0) \mid D_i = 1] = \mathbb{E}_\omega[Y_{i,t} \mid D_i = 1] - \underbrace{\mathbb{E}_\omega[Y_{i,t}(0) \mid D_i = 1]}_{\text{counterfactual}}. \tag{2}$$

We want $ATT(2014)$. Unweighted, it is the average effect of Medicaid expansion on 2014 mortality among **counties** in 2014-expansion states. With population weights, it is the average effect for **adults** in those counties. These are different target parameters.

### 3.2 Identifying assumptions: parallel trends

**Assumption NA (no anticipation).** For treated units in pretreatment periods, $Y_{i,t}(1) = Y_{i,t}(0)$. So $Y_{i,2013} = Y_{i,2013}(0)$ and $ATT(2013) = 0$.

**Assumption PT (2×2 parallel trends).** Without treatment, average outcomes would have changed by the same amount in both groups:

$$\mathbb{E}_\omega[Y_{i,2014}(0) \mid D_i = 1] - \mathbb{E}_\omega[Y_{i,2013}(0) \mid D_i = 1] = \mathbb{E}_\omega[Y_{i,2014}(0) \mid D_i = 0] - \mathbb{E}_\omega[Y_{i,2013}(0) \mid D_i = 0]. \tag{3}$$

Under NA and PT, the missing counterfactual equals the treated group's 2013 mean plus the comparison group's 2013→2014 change. Substituting it into (2) gives the **2×2 DiD estimand**:

$$ATT(2014) = \underbrace{\Big(\mathbb{E}_\omega[Y_{i,2014} \mid D_i = 1] - \mathbb{E}_\omega[Y_{i,2013} \mid D_i = 1]\Big)}_{\text{average change for } D_i = 1} - \underbrace{\Big(\mathbb{E}_\omega[Y_{i,2014} \mid D_i = 0] - \mathbb{E}_\omega[Y_{i,2013} \mid D_i = 0]\Big)}_{\text{average change for } D_i = 0}. \tag{5}$$

### 3.3 Estimation: four means or one regression?

#### Four means (Table 2)

Replace each population expectation in (5) with its sample mean:

$$\widehat{ATT}(2014) = \left(\bar{Y}_{D=1,\,t=2014} - \bar{Y}_{D=1,\,t=2013}\right) - \left(\bar{Y}_{D=0,\,t=2014} - \bar{Y}_{D=0,\,t=2013}\right). \tag{6}$$

The three cells below compute the four means, then the change within each group (Table 2, row 3), then the difference of those changes. The paper reports 0.1 deaths per 100,000 (unweighted).

In [ ]:
keep_row = (
    df['treat_year'].isin([0, 2014])
    & df['year'].isin([2013, 2014])
)

did_df = df.loc[keep_row].copy()

did_df['treat'] = (did_df['treat_year'] == 2014).astype(int)
did_df['post']  = (did_df['year'] == 2014).astype(int)

mean_df = did_df.groupby(['treat', 'post'])['crude_rate'].mean()
mean_df


In [ ]:
mean_df.reset_index().groupby('treat').diff().round(1)

In [ ]:
mean_df.reset_index().groupby('treat').diff().dropna().diff()

#### One regression (Table 3)

The same number is the least squares estimate of $\beta^{2\times2}$ in

$$Y_{i,t} = \beta_0 + \beta_1 \mathbb{1}\{D_i = 1\} + \beta_2 \mathbb{1}\{t = 2014\} + \beta^{2\times2}\left(\mathbb{1}\{D_i = 1\} \times \mathbb{1}\{t = 2014\}\right) + \varepsilon_{i,t}, \tag{7}$$

estimated on 2013 and 2014 data only. Each of the four means is a sum of coefficients:

| | $D_i = 0$ | $D_i = 1$ |
|---|---|---|
| 2013 | $\hat\beta_0$ | $\hat\beta_0 + \hat\beta_1$ |
| 2014 | $\hat\beta_0 + \hat\beta_2$ | $\hat\beta_0 + \hat\beta_1 + \hat\beta_2 + \hat\beta^{2\times2}$ |

Plugging these into (6) gives $\widehat{ATT}(2014) = \hat\beta^{2\times2}$ exactly. With a balanced panel the estimate is also identical if the group and post dummies are replaced by county and year fixed effects (the TWFE regression $Y_{i,t} = \theta_t + \eta_i + \beta^{2\times2} D_{i,t} + \varepsilon_{i,t}$).

Following the paper's sampling-based view of uncertainty, standard errors are clustered by county. Below, the first regression is Table 3, column 1. The `pyfixest` regression with county and year fixed effects is column 2.

In [ ]:
# note that in patsy `treat * post` expands to `treat + post + treat * post`
did_mod = smf.wls('crude_rate ~ treat * post', data=did_df)
did_res = did_mod.fit(cov_type='cluster', cov_kwds={'groups': did_df['county_code']})
did_res.summary(slim=True)

In [ ]:
# Function to generate significance stars
def get_significance_stars(pval):
    if pval < 0.001:
        return '****'
    elif pval < 0.01:
        return '***'
    elif pval < 0.05:
        return '**'
    elif pval < 0.1:
        return '*'
    else:
        return ''

# combining the above into a function
def get_result_array(res, keep_vars=None, name=None, add_nobs=False, show_stars=False, round_to=3):
    if keep_vars is None:
        keep_vars = res.params.index

    if show_stars:
        coef_series = (
            res.params[keep_vars].round(round_to).astype(str)           # format the estimated parameters as rounded strings
            +                                                        # use addition to add stars (if any)
            res.pvalues.apply(get_significance_stars)[keep_vars] # generate significance stars
        )
    else:
        coef_series = (
            res.params[keep_vars].round(round_to).astype(str)           # format the estimated parameters as rounded strings
        )
    coef_series.name = 'coef'  # naming series will be helpful when this is a dataframe
    tvalue_series = '(' + res.bse[keep_vars].round(round_to).astype(str) + ')' # add parentheses around formatted t-values
    tvalue_series.name = 'se'

    series_list = [coef_series, tvalue_series]
    if add_nobs:
        obs_series = pd.Series([f'{res.nobs:.0f}'], index=['_nobs'], name='obs')
        series_list = series_list + [obs_series]

    ols_res_series = pd.concat(series_list).sort_index(ascending=False)
    if name is not None:
        ols_res_series.name = name

    # reorder index
    # index_order = keep_vars + ['_nobs']
    # ols_res_series = ols_res_series.loc[index_order]
    
    return ols_res_series

ols_array = get_result_array(did_res, round_to=1)
ols_array.sort_index()


In [ ]:
fe_res = pf.feols('crude_rate ~ treat:post | county_code + year', data=did_df,
                  vcov={'CRV1': 'county_code'})
pf.etable(fe_res)

## 4. Incorporating Covariates into 2×2 DiD

Parallel trends can fail if expansion and non-expansion counties differ in characteristics that predict mortality *trends*. For example, poorer and richer counties may have had different mortality trends even without Medicaid expansion. This section uses covariates to (i) judge whether unconditional parallel trends is plausible, and (ii) identify the same $ATT(2014)$ under a weaker assumption.

### 4.1 Covariate balance: are unconditional parallel trends plausible? (Table 4)

We compare covariates across groups using the **normalized difference**:

$$\text{Norm. Diff} = \frac{\bar{X}_{T} - \bar{X}_{C}}{\sqrt{(S^2_{T} + S^2_{C})/2}},$$

where $\bar X$ and $S^2$ are the sample mean and variance in the treatment ($T$) and comparison ($C$) groups. As a rule of thumb, $|\text{Norm. Diff}| > 0.25$ signals worrisome imbalance (Imbens and Rubin 2015).

We check balance in two ways:

- **Baseline levels:** $\mathbb{E}_\omega[X_{i,2013} \mid D_i = 1]$ vs. $\mathbb{E}_\omega[X_{i,2013} \mid D_i = 0]$. This is the top panel of Table 4 and the first cell below.
- **Changes:** $\mathbb{E}_\omega[\Delta X_{i,2014} \mid D_i = 1]$ vs. $\mathbb{E}_\omega[\Delta X_{i,2014} \mid D_i = 0]$, where $\Delta X_{i,2014} = X_{i,2014} - X_{i,2013}$. This is the bottom panel and the second cell.

DiD does **not** automatically "difference out" baseline differences. Imbalance in baseline levels still threatens parallel trends if those covariates are related to trends in $Y_{i,t}(0)$.

In [ ]:
covariates = ['perc_female', 'perc_white', 'perc_hispanic', 'unemp_rate_pc', 'median_income_k', 'poverty_rate']

check_df = did_df.loc[did_df['year'] == 2013].copy() 
group_df = check_df.groupby('treat')[covariates]
mean_by_treat = group_df.mean().T
std_by_treat = group_df.std().T

mean_by_treat['norm_diff'] = (
    (mean_by_treat[1] - mean_by_treat[0]) 
    / np.sqrt((std_by_treat[1]**2 + std_by_treat[0]**2) / 2)
)

print(tabulate(mean_by_treat, headers=['Non-adopt', 'Adopt', 'Norm. diff'], floatfmt='.2f'))


In [ ]:
diff_df = did_df.sort_values('year').groupby('county_code')[covariates].diff().dropna()
diff_df['treat'] = did_df.loc[diff_df.index, 'treat']
# print(tabulate(balance_table(diff_df), headers='keys', floatfmt='.2f'))

check_df = diff_df.copy() 
group_df = check_df.groupby('treat')[covariates]
mean_by_treat = group_df.mean().T
std_by_treat = group_df.std().T

mean_by_treat['norm_diff'] = (
    (mean_by_treat[1] - mean_by_treat[0]) 
    / np.sqrt((std_by_treat[1]**2 + std_by_treat[0]**2) / 2)
)

print(tabulate(mean_by_treat, headers=['Non-adopt', 'Adopt', 'Norm. diff'], floatfmt='.2f'))

### 4.2 Identification under conditional parallel trends

Let $X_i$ be a vector of observed determinants of changes in $Y_{i,t}(0)$. Here, $X_i$ is the 2013 covariate values.

**Assumption CPT (2×2 conditional parallel trends).** Parallel trends holds between treated and comparison counties *with the same covariate values*:

$$\mathbb{E}_\omega[Y_{i,2014}(0) - Y_{i,2013}(0) \mid X_i, D_i = 1] = \mathbb{E}_\omega[Y_{i,2014}(0) - Y_{i,2013}(0) \mid X_i, D_i = 0]. \tag{8}$$

**Assumption SO (strong overlap).** For some $\epsilon > 0$, $\;\epsilon < P_\omega[D_i = 1 \mid X_i] < 1 - \epsilon$. Every covariate value has both treated and comparison counties.

Under NA, CPT, and SO, the target parameter is identified:

$$ATT(2014) = \mathbb{E}_\omega[\Delta Y_{i,2014} \mid D_i = 1] - \mathbb{E}_\omega\Big[\,\mathbb{E}_\omega[\Delta Y_{i,2014} \mid X_i, D_i = 0] \;\Big|\; D_i = 1\Big], \tag{9}$$

where $\Delta Y_{i,2014} = Y_{i,2014} - Y_{i,2013}$. In words: take the treated group's average change, and subtract the comparison group's average change *at each covariate value*, averaged over the treated group's covariate distribution.

The target parameter is still $ATT(2014)$ from Section 3. Only the identifying assumption has changed.

### 4.3 DiD estimation with covariates: TWFE (Table 5)

The most common practice is to add covariates to the TWFE regression. The paper considers two specifications:

$$Y_{i,t} = \theta_t + \eta_i + \beta_{\text{treat}}\, D_{i,t} + X_{i,t}'\beta_{\text{covs}} + e_{i,t} \tag{10}$$

$$Y_{i,t} = \theta_t + \eta_i + \beta_{\text{treat},2}\, D_{i,t} + \left(\mathbb{1}\{t = 2014\}\,X_{i,2013}\right)'\beta_{\text{covs},2} + e_{i,t} \tag{11}$$

With two periods, these are equivalent to regressing $\Delta Y_{i,2014}$ on $D_i$ and either the **change** in the covariates, $\Delta X_{i,2014}$ (eq. 10), or their **2013 level**, $X_{i,2013}$ (eq. 11). Controlling for annual poverty rates really means controlling for poverty *changes*.

**Caveat.** Under CPT, neither $\beta_{\text{treat}}$ nor $\beta_{\text{treat},2}$ generally equals $ATT(2014)$. They recover it only if the treatment effect is the same at every covariate value (and the linear model is correct). Otherwise they are weighted averages of conditional ATTs, and the weights can be negative (Caetano and Callaway 2024). Equation (10) also has a "bad control" problem if the 2014 covariates were themselves affected by the expansion.

The table below matches Table 5, columns 1–3: no covariates, 2013 levels × post (eq. 11), and time-varying covariates (eq. 10).

In [ ]:
covariates = ['perc_female', 'perc_white', 'perc_hispanic', 'unemp_rate_pc', 'poverty_rate', 'median_income_k']

# 2013 covariate levels, constant within county
for c in covariates:
    did_df[c + '_2013'] = (did_df[c].where(did_df['year'] == 2013)
                           .groupby(did_df['county_code']).transform('max'))

fe = ' | county_code + year'
x13 = ' + '.join(f'{c}_2013:post' for c in covariates)  # col 2: 2013 levels x post
xt  = ' + '.join(covariates)                            # col 3: time-varying, which the FE turn into changes

fits = [
    pf.feols('crude_rate ~ treat:post' + fe,              data=did_df, vcov={'CRV1': 'county_code'}),
    pf.feols(f'crude_rate ~ treat:post + {x13}' + fe,     data=did_df, vcov={'CRV1': 'county_code'}),
    pf.feols(f'crude_rate ~ treat:post + {xt}' + fe,      data=did_df, vcov={'CRV1': 'county_code'}),
]
pf.etable(fits, keep='treat:post')

### 4.4 DiD estimators with covariates that target $ATT(2014)$

Instead of starting from a regression, we can start from the identification result (9) and estimate each piece. Each estimator below targets the same $ATT(2014)$ under the same assumptions (NA, CPT, SO). They differ in which **working model** they use for the unknown functions in (9).

The two working models (Table 6):

- **Outcome regression** for the comparison group's expected change, $\mu_{\Delta,D=0}(X_i) = X_i'\beta_{D=0}$. This approximates $\mathbb{E}[\Delta Y_{i,2014} \mid X_i, D_i = 0]$. Estimate it by regressing $\Delta Y_{i,2014}$ on $X_{i,2013}$ using **only comparison counties**, then predict $\hat\mu_{\Delta,D=0}(X_i)$ for every county.
- **Propensity score**, $p(X_i) = P(D_i = 1 \mid X_i)$. Estimate it with a logit of $D_i$ on $X_{i,2013}$ using **all counties**, and call the fitted values $\hat\pi(X_i)$.

The cell below estimates both, matching the unweighted columns (1 and 2) of Table 6.

In [ ]:
# one row per county: 2013 covariates + change in mortality
cs = did_df[did_df['year'] == 2013].copy()
cs['delta_y'] = (
    did_df[did_df['year'] == 2014].set_index('county_code')['crude_rate']
    .reindex(cs['county_code']).values 
    - cs['crude_rate'].values
)

x13 = ' + '.join(f'{c}_2013' for c in covariates)

or_res = pf.feols(f'delta_y ~ {x13}', data=cs[cs['treat'] == 0], vcov='hetero')      # outcome regression
ps_res = pf.feglm(f'treat ~ {x13}', data=cs, family='logit', vcov='hetero')          # propensity score

pf.etable([or_res, ps_res])

#### Regression adjustment, IPW, and doubly robust estimators (Table 7)

**Regression adjustment (RA).** Plug the fitted outcome model into the inner expectation in (9):

$$\widehat{ATT}_{ra}(2014) = \frac{\sum_{i=1}^n D_i\left(\Delta Y_{i,2014} - \hat\mu_{\Delta,D=0}(X_i)\right)}{\sum_{i=1}^n D_i}. \tag{12}$$

**Inverse probability weighting (IPW).** Reweight comparison counties so that their covariate distribution matches the treated group's. Comparison counties that "look treated" (high $\hat\pi(X_i)$) get more weight:

$$\widehat{ATT}_{ipw}(2014) = \frac{1}{n}\sum_{i=1}^n \left(\hat w_{D=1}(D_i) - \hat w_{D=0}(D_i, X_i)\right)\Delta Y_{i,2014}, \tag{15}$$

$$\hat w_{D=1}(D) = \frac{D}{\frac1n\sum_{i} D_i}, \qquad \hat w_{D=0}(D, X) = \frac{(1-D)\,\hat\pi(X)/(1-\hat\pi(X))}{\frac1n\sum_{i} (1-D_i)\,\hat\pi(X_i)/(1-\hat\pi(X_i))}. \tag{16}$$

**Doubly robust (DR).** Combine the two by applying the IPW weights to the residual from the outcome model:

$$\widehat{ATT}_{dr}(2014) = \frac{1}{n}\sum_{i=1}^n \left(\hat w_{D=1}(D_i) - \hat w_{D=0}(D_i, X_i)\right)\left(\Delta Y_{i,2014} - \hat\mu_{\Delta,D=0}(X_i)\right). \tag{18}$$

DR is consistent if *either* the outcome model *or* the propensity score model is correctly specified. The paper recommends it over RA or IPW alone.

**Mapping to the code.** In `drdid()` below, `resid` is $\Delta Y_{i,2014} - \hat\mu_{\Delta,D=0}(X_i)$, and `ctrl_wt` is the unnormalized comparison weight $(1-D)\,\hat\pi/(1-\hat\pi)$. Dividing by `ctrl_wt.sum()` does the normalization in (16).

**Inference.** These are two-step estimators, so the standard errors must account for the estimation error in the working models. The paper uses analytic standard errors. Here we use a nonparametric bootstrap that resamples counties, so our standard errors will be close to Table 7's but not identical.

In [ ]:
# outcome regression
x13 = ' + '.join(f'{c}_2013' for c in covariates)
reg_str = f'delta_y ~ {x13}'
print(reg_str)
or_mod = pf.feols(reg_str, data=cs[cs['treat'] == 0], vcov='hetero') 
# or_res = or_mod.fit()
pred_change = or_res.predict(cs)


resid = cs['delta_y'] - pred_change
regression_adjustment = resid.loc[cs.loc[cs['treat'] == 1].index].mean()
print(f'Regression adjustment ATT: {regression_adjustment:.2f}')

In [ ]:
# IPW example
x13 = ' + '.join(f'{c}_2013' for c in covariates)

ps_mod = smf.logit(f'treat ~ {x13}', data=cs)
ps_res = ps_mod.fit(disp=0)
pscore = ps_res.predict(cs)

ps_res.summary()

ctrl_wt = pscore * (1 - cs['treat']) / (1 - pscore)

ipw_att = cs['delta_y'].loc[cs['treat'] == 1].mean() - (ctrl_wt * cs['delta_y']).sum() / ctrl_wt.sum()

print(f'IPW ATT: {ipw_att:.2f}')

In [ ]:
def drdid(data):
    outcome = data['delta_y']
    treated = data['treat']

    # outcome model on control counties (Table 6 col 1)
    or_mod = smf.ols(f'delta_y ~ {x13}', data=data[treated == 0])
    or_res = or_mod.fit()
    pred_change = or_res.predict(data)

    # propensity score model (Table 6 col 2)
    ps_mod = smf.logit(f'treat ~ {x13}', data=data)
    ps_res = ps_mod.fit(disp=0)
    pscore = ps_res.predict(data)

    # weights for control counties
    ctrl_wt = pscore * (1 - treated) / (1 - pscore)
    resid = outcome - pred_change

    return pd.Series({
        'Regression':    resid[treated == 1].mean(),
        'IPW':           outcome[treated == 1].mean() - (ctrl_wt * outcome).sum() / ctrl_wt.sum(),
        'Doubly Robust': resid[treated == 1].mean() - (ctrl_wt * resid).sum() / ctrl_wt.sum(),
    })

att_est = drdid(cs)

# bootstrap SEs: resample counties
boot_df = pd.DataFrame([
    drdid(cs.sample(frac=1, replace=True, random_state=seed).reset_index(drop=True))
    for seed in range(999)
])

results_df = pd.DataFrame({'ATT': att_est, 'SE': boot_df.std()})
print(tabulate(results_df, headers='keys', floatfmt='.2f'))

#### Checking overlap (Figure 1)

IPW and DR estimates become unstable when comparison counties have propensity scores near 1, because $\hat\pi/(1-\hat\pi)$ explodes. Assumption SO requires the two groups' propensity score distributions to overlap. The paper finds that few comparison counties have very high scores, and that the comparison counties' scores lie within the range of the expansion counties' scores.

In [ ]:
ps_mod = smf.logit(f'treat ~ {x13}', data=cs)
ps_res = ps_mod.fit(disp=0)
cs['pscore'] = ps_res.predict(cs)

fig, ax = plt.subplots()
ax.hist(cs.loc[cs['treat'] == 1, 'pscore'], bins=30, density=True, histtype='step', color='red', label='Expansion')
ax.hist(cs.loc[cs['treat'] == 0, 'pscore'], bins=30, density=True, histtype='step', color='blue', label='Non-Expansion')
ax.set_xlabel('Propensity Score')
ax.set_ylabel('Density')
ax.legend()

## 5. DiD Designs with Multiple Time Periods

### 5.1 Simple event study (2×T)

We keep the same two groups (2014 expanders vs. states that had not expanded by 2019) but now use every year from 2009 to 2019. The treatment date is $g = 2014$, so the reference period is $g - 1 = 2013$. **This section uses population weights, $\omega_i = W_i$** (`set_wt`), as in the paper's Figure 3.

**Target parameters.** There is a separate $ATT(t)$ for each post-period $t \geq g$, defined exactly as in (2). Together they show how the effect evolves over time.

**Assumption PT-ES (2×T parallel trends).** Parallel trends holds between $g-1$ and *every* post-period:

$$\mathbb{E}_\omega[Y_{i,t}(0) - Y_{i,g-1}(0) \mid D_i = 1] = \mathbb{E}_\omega[Y_{i,t}(0) - Y_{i,g-1}(0) \mid D_i = 0] \quad \text{for all } t \geq g. \tag{19}$$

Learning about longer-run effects requires parallel trends to hold over a longer horizon.

**Estimator.** Each $ATT(t)$ is a 2×2 DiD comparing year $t$ to 2013:

$$\widehat{ATT}(t) = \left(\bar Y_{\omega,D=1,t} - \bar Y_{\omega,D=1,g-1}\right) - \left(\bar Y_{\omega,D=0,t} - \bar Y_{\omega,D=0,g-1}\right). \tag{20}$$

**Pre-trends.** The same comparison for pre-periods $t = g-k$ (with $k > 1$) gives

$$\tau_{-k} = \mathbb{E}_\omega[Y_{i,g-k} - Y_{i,g-1} \mid D_i = 1] - \mathbb{E}_\omega[Y_{i,g-k} - Y_{i,g-1} \mid D_i = 0].$$

Under NA, these measure differences in untreated trends *before* treatment. They are a useful but indirect check. PT-ES is a restriction on post-period trends in $Y_{i,t}(0)$, which we never observe.

**One regression.** All the $\widehat{ATT}(t)$ and $\hat\tau_{-k}$ come from a single TWFE regression that interacts the treatment-group dummy with year dummies and omits $t = g-1$:

$$Y_{i,t} = \theta_t + \eta_i + \sum_{k=1}^{g-2}\beta_k\left(\mathbb{1}\{G_i = g\}\cdot\mathbb{1}\{t=k\}\right) + \sum_{k=g}^{T}\beta_k\left(\mathbb{1}\{G_i = g\}\cdot\mathbb{1}\{t=k\}\right) + \varepsilon_{i,t}. \tag{21}$$

For $t \geq g$, $\hat\beta_t = \widehat{ATT}(t)$. For $t < g-1$, $\hat\beta_t$ is the pre-trend estimate $\hat\tau_{-(g-t)}$. In the code, `i(year, treat, ref=2013)` builds these interactions. The plot's x-axis is event time, $e = t - g$.

**Inference.** The figure shows two kinds of 95% intervals:

- **pointwise** (black) intervals cover each coefficient separately;
- **uniform** (red) bands cover the whole event-study path at once. They use a larger sup-t critical value to account for testing many coefficients. `confint(joint=True)` computes that critical value by simulation, hence the `seed`.

In [ ]:
es_df = df[df['treat_year'].isin([0, 2014])].copy()
es_df['treat'] = (es_df['treat_year'] == 2014).astype(int)

es_res = pf.feols('crude_rate ~ i(year, treat, ref=2013) | county_code + year',
                  data=es_df, weights='set_wt', vcov={'CRV1': 'county_code'})

coefs = es_res.coef().values
ci_pointwise = es_res.confint().values
ci_uniform = es_res.confint(joint=True, seed=20240924).values
event_time = np.array([year for year in sorted(es_df['year'].unique()) if year != 2013]) - 2014

fig, ax = plt.subplots()
ax.vlines(event_time, ci_uniform[:, 0], ci_uniform[:, 1], color='red', label='Uniform CI')
ax.vlines(event_time, ci_pointwise[:, 0], ci_pointwise[:, 1], color='black', label='Pointwise CI')
ax.scatter(np.append(event_time, -1), np.append(coefs, 0), color='black', zorder=3)
ax.axhline(0, linestyle='--', color='gray')
ax.axvline(-1, linestyle='--', color='gray')
ax.set_xlabel('Event-Time')
ax.set_ylabel('Treatment Effect\nMortality Per 100,000')
ax.legend()